# Notebook 01: Data Cleaning and Preprocessing

This notebook loads the raw bank churn dataset, performs cleaning and preprocessing, and saves outputs for downstream analysis/modeling.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("pandas version:", pd.__version__)

pandas version: 3.0.2


In [2]:
def find_project_root(start: Path) -> Path:
    """Walk upward until a folder containing Data/Raw is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "Raw").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing Data/Raw")

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_PATH = PROJECT_ROOT / "Data" / "Raw" / "Bank Customer Churn Prediction.csv"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
FEATURES_DIR = PROJECT_ROOT / "Data" / "Features"
MODELS_DIR = PROJECT_ROOT / "Models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw file:", RAW_PATH)
print("Raw file exists:", RAW_PATH.exists())

Project root: /Users/ganenthraravindran/Desktop/Analytics in Practise
Raw file: /Users/ganenthraravindran/Desktop/Analytics in Practise/Data/Raw/Bank Customer Churn Prediction.csv
Raw file exists: True


In [3]:
df_raw = pd.read_csv(RAW_PATH)

print("Shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.info())

Shape: (10000, 12)


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10000 non-null  int64  
 1   credit_score      10000 non-null  int64  
 2   country           10000 non-null  str    
 3   gender            10000 non-null  str    
 4   age               10000 non-null  int64  
 5   tenure            10000 non-null  int64  
 6   balance           10000 non-null  float64
 7   products_number   10000 non-null  int64  
 8   credit_card       10000 non-null  int64  
 9   active_member     10000 non-null  int64  
 10  estimated_salary  10000 non-null  float64
 11  churn             10000 non-null  int64  
dtypes: float64(2), int64(8), str(2)
memory usage: 937.6 KB


None

In [4]:
# Standardize column names and remove exact duplicate rows
df = df_raw.copy()
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("[^a-zA-Z0-9_]", "", regex=True)
)

before_dupes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after_dupes = len(df)

print("Rows removed as duplicates:", before_dupes - after_dupes)
print("Updated shape:", df.shape)
print("Columns:")
display(pd.Series(df.columns, name="column_name"))

Rows removed as duplicates: 0
Updated shape: (10000, 12)
Columns:


0          customer_id
1         credit_score
2              country
3               gender
4                  age
5               tenure
6              balance
7      products_number
8          credit_card
9        active_member
10    estimated_salary
11               churn
Name: column_name, dtype: str

In [5]:
# Missing-value summary
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_report = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
display(missing_report[missing_report["missing_count"] > 0])

,missing_count,missing_pct


In [6]:
# Define target, remove leakage-prone identifier columns, and split before fitting transforms
target_candidates = ["churn", "exited", "attrition", "target", "label"]
target_col = next((c for c in target_candidates if c in df.columns), None)

if target_col is None:
    raise ValueError("No target column found. Expected one of: churn, exited, attrition, target, label")

id_like_cols = [c for c in df.columns if c.endswith("_id") or c in {"id", "customerid", "customer_id"}]
feature_cols = [c for c in df.columns if c != target_col and c not in id_like_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique(dropna=True) <= 20 else None,
    shuffle=True,
)

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Detected target column:", target_col)
print("Dropped ID-like columns:", id_like_cols if id_like_cols else "None")
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Numeric columns:", len(numeric_cols), "| Categorical columns:", len(categorical_cols))

Detected target column: churn
Dropped ID-like columns: ['customer_id']
Train shape: (8000, 10) | Test shape: (2000, 10)
Numeric columns: 8 | Categorical columns: 2


In [7]:
# Build leakage-safe preprocessing pipeline fitted on train only
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ],
    remainder="drop",
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Train processed shape:", X_train_processed_df.shape)
print("Test processed shape:", X_test_processed_df.shape)

Train processed shape: (8000, 11)
Test processed shape: (2000, 11)


In [8]:
# Class balance check and final train/test modeling tables
train_churn_rate = y_train.mean()
test_churn_rate = y_test.mean()

train_model_df = pd.concat([X_train_processed_df, y_train.rename(target_col)], axis=1)
test_model_df = pd.concat([X_test_processed_df, y_test.rename(target_col)], axis=1)

print(f"Train churn rate: {train_churn_rate:.4f}")
print(f"Test churn rate: {test_churn_rate:.4f}")
print("Train modeling table shape:", train_model_df.shape)
print("Test modeling table shape:", test_model_df.shape)

Train churn rate: 0.2037
Test churn rate: 0.2035
Train modeling table shape: (8000, 12)
Test modeling table shape: (2000, 12)


In [9]:
cleaned_path = PROCESSED_DIR / "bank_customer_churn_cleaned.csv"
train_features_path = FEATURES_DIR / "bank_customer_churn_train_features.csv"
test_features_path = FEATURES_DIR / "bank_customer_churn_test_features.csv"
preprocessor_path = MODELS_DIR / "bank_customer_churn_preprocessor.joblib"

df.to_csv(cleaned_path, index=False)
train_model_df.to_csv(train_features_path, index=False)
test_model_df.to_csv(test_features_path, index=False)
dump(preprocessor, preprocessor_path)

print("Saved cleaned data to:", cleaned_path)
print("Saved train features to:", train_features_path)
print("Saved test features to:", test_features_path)
print("Saved preprocessor to:", preprocessor_path)

Saved cleaned data to: /Users/ganenthraravindran/Desktop/Analytics in Practise/Data/Processed/bank_customer_churn_cleaned.csv
Saved train features to: /Users/ganenthraravindran/Desktop/Analytics in Practise/Data/Features/bank_customer_churn_train_features.csv
Saved test features to: /Users/ganenthraravindran/Desktop/Analytics in Practise/Data/Features/bank_customer_churn_test_features.csv
Saved preprocessor to: /Users/ganenthraravindran/Desktop/Analytics in Practise/Models/bank_customer_churn_preprocessor.joblib


In [10]:
display(train_model_df.head())
display(test_model_df.head())

,num__credit_score,num__age,num__tenure,num__balance,num__products_number,num__credit_card,num__active_member,num__estimated_salary,cat__country_Germany,cat__country_Spain,cat__gender_Male,churn
2151,1.058568,1.715086,0.684723,-1.226059,-0.910256,0.641042,-1.030206,1.042084,0.0,0.0,1.0,1
8392,0.913626,-0.659935,-0.696202,0.413288,-0.910256,0.641042,-1.030206,-0.623556,1.0,0.0,1.0,1
5006,1.079274,-0.184931,-1.731895,0.601687,0.808830,0.641042,0.970680,0.308128,1.0,0.0,0.0,0
4117,-0.929207,-0.184931,-0.005739,-1.226059,0.808830,0.641042,-1.030206,-0.290199,0.0,0.0,1.0,0
7182,0.427035,0.955079,0.339492,0.548318,0.808830,-1.559960,0.970680,0.135042,1.0,0.0,1.0,0


,num__credit_score,num__age,num__tenure,num__balance,num__products_number,num__credit_card,num__active_member,num__estimated_salary,cat__country_Germany,cat__country_Spain,cat__gender_Male,churn
5702,-0.680735,-0.279932,0.684723,-1.226059,0.808830,0.641042,-1.030206,-0.095021,0.0,0.0,1.0,0
3667,-1.301915,-0.564935,-0.350971,0.877113,0.808830,-1.559960,-1.030206,-0.778941,1.0,0.0,1.0,0
1617,-0.970619,0.100072,-0.350971,-1.226059,0.808830,-1.559960,0.970680,0.099469,0.0,1.0,0.0,0
5673,-0.121674,-0.469934,-0.005739,1.011458,0.808830,-1.559960,-1.030206,-1.147374,0.0,1.0,1.0,0
4272,-0.111321,-0.469934,-0.696202,0.023204,-0.910256,0.641042,0.970680,1.200283,0.0,1.0,0.0,0
